In [19]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import RobustScaler 
from sklearn.metrics import mean_absolute_error
import os

In [20]:
df = pd.read_csv('data/final_data_outliers_clean_Iso.csv')
df.shape

(8645, 34)

In [21]:
# # df = pd.read_csv('data/final_data_outliers_clean_iqr.csv')

# #Reshape the data into a 3D tensor with shape (num_samples, 2, 15)
# class ImpedanceDataset(Dataset):
#     def __init__(self, csv_file, is_train = True, scaler_mag = None, scaler_phase = None, label_encoder = None):
#         #read the CSV file
#         df = pd.read_csv(csv_file)

#         mag_cols = [f'Magnitude_{i}' for i in range(15)]
#         phase_cols = [f'Phase_{i}' for i in range(15)]

#         X_mag = df[mag_cols].values
#         X_phase = df[phase_cols].values

#         Y_targets = df[['Value_R', 'Value_C']].values

#         #Scaler for Magnitude and Phase
#         #true (if) for training, false (else) for testing
#         if is_train: 
#             self.scaler_mag = StandardScaler()
#             self.scaler_phase = StandardScaler()
#             self.scaler_target = StandardScaler()

#             X_mag_scaled = self.scaler_mag.fit_transform(X_mag)
#             X_phase_scaled = self.scaler_phase.fit_transform(X_phase)
#             Y_targets_scaled = self.scaler_target.fit_transform(Y_targets)

#         else: 
#             self.scaler_mag = scaler_mag
#             self.scaler_phase = scaler_phase
#             self.scaler_target = scaler_target

#             X_mag_scaled = self.scaler_mag.transform(X_mag)
#             X_phase_scaled = self.scaler_phase.transform(X_phase)
#             Y_targets_scaled = self.scaler_target.transform(Y_targets)
        
#         #Merge and reshape the data into tensors
#         # X_mag_scaled shape: (num_samples, 15) -> new shape: (num_samples, 1, 15)
#         X_mag_tensor = np.expand_dims(X_mag_scaled, axis=1)
#         X_phase_tensor = np.expand_dims(X_phase_scaled, axis=1)

#         #combine 2 slices of mag and phase
#         #into 1 tensor with shape (num_samples, 2, 15)
#         self.X_data = np.concatenate((X_mag_tensor, X_phase_tensor), axis=1)

#         self.X_data = torch.tensor(self.X_data, dtype=torch.float32)
#         self.targets = torch.tensor(Y_targets_scaled, dtype=torch.float32)

#     def __len__(self):
#         return len(self.X_data)

#     def __getitem__(self, idx):
#         return self.X_data[idx], self.targets[idx]

In [22]:
class ImpedanceDataset(Dataset):
    def __init__(self, X_mag, X_phase, Y_targets):
        X_mag_tensor = np.expand_dims(X_mag, axis=1)
        X_phase_tensor = np.expand_dims(X_phase, axis=1)
        self.X_data = np.concatenate((X_mag_tensor, X_phase_tensor), axis=1)
        
        self.X_data = torch.tensor(self.X_data, dtype=torch.float32)
        self.targets = torch.tensor(Y_targets, dtype=torch.float32)

    def __len__(self):
        return len(self.X_data)

    def __getitem__(self, idx):
        return self.X_data[idx], self.targets[idx]

## 1D-CNN + FNN design 

In [23]:
# class CNN1D(nn.Module):
#     def __init__(self, num_classes):
#         super(CNN1D, self).__init__()

#         """
#         Block 1: 
#         Input shape: (batch_size, 2, 15)

#         Sequence length (L) = 15: 15 points of Magnitude and Phase from index 0 -> 14
#         Channels (C) = 2: including Magnitude and Phase
#         Tensor shape: N x C x L = N x 2 x 15
#         """

#         self.conv_block1 = nn.Sequential(
#             nn.Conv1d(in_channels = 2, out_channels = 16, kernel_size = 3, padding = 1),
#             nn.BatchNorm1d(16),
#             nn.ReLU(),
#             nn.MaxPool1d(kernel_size = 2) # Reduces the sequence length from 15 to 7
#         )


#         """
#         Block 2:
#         Input shape: (batch_size, 16, 7)

#         Sequence length (L) = 7: 7 points of Magnitude and Phase from index 0 -> 6
#         Channels (C) = 16: output channels from the previous convolutional layer
#         Tensor shape: N x C x L = N x 16 x 7
#         """

#         self.conv_block2 = nn.Sequential(
#             nn.Conv1d(in_channels = 16, out_channels = 32, kernel_size = 3, padding = 1),
#             nn.BatchNorm1d(32),
#             nn.ReLU(),
#             #Avoid overfitting by using AdaptiveAvgPool1d to reduce the sequence length to 1

#             nn.AdaptiveAvgPool1d(1) # Reduces the sequence length from 7 to 1
#         )
    
#         """
#         Block 3:
#         Input shape: (batch_size, 32, 1)

#         Sequence length (L) = 1: 1 point of Magnitude and Phase from index 0
#         Channels (C) = 32: output channels from the previous convolutional layer
#         Tensor shape: N x C x L = N x 32 x 1
#         """

#         self.fnn = nn.Sequential(
#             nn.Flatten(), # Flattens the tensor to shape (batch_size, 32)
#             nn.Linear(32, 64),
#             # nn.Linear(32, 16),
#             nn.Dropout(0.2),
#             nn.ReLU(),
#             nn.Linear(64, num_classes) # Output layer with 2 neurons for Value_R and Value_C
#         )

#     def forward(self, x):
#         x = self.conv_block1(x)
#         x = self.conv_block2(x)
#         out = self.fnn(x)
#         return out

In [24]:
class CNN1D_Robust(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Sequential(
            nn.Conv1d(in_channels=2, out_channels=8, kernel_size=3, padding=1),
            nn.BatchNorm1d(8),
            nn.ReLU(),
            nn.Dropout1d(p = 0.1),
            nn.MaxPool1d(kernel_size = 2)
        )

        self.conv2 = nn.Sequential(
                nn.Conv1d(in_channels=8, out_channels=16, kernel_size=3, padding=1),
                nn.BatchNorm1d(16),
                nn.ReLU(),
                nn.Dropout1d(p = 0.1),
                nn.AdaptiveAvgPool1d(output_size = 1)
        )

        self.fnn = nn.Sequential(
                nn.Flatten(),
                nn.Linear(16, 32),
                nn.ReLU(),
                nn.Dropout(p = 0.4),
                nn.Linear(32, 2)
            )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        return self.fnn(x)

In [25]:
# def main():
#     data_path = 'data/final_data_outliers_clean_Iso.csv'
#     batch_size = 64
#     epochs = 100
#     learning_rate = 0.001

#     device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#     # Load the dataset
#     full_dataset = ImpedanceDataset(data_path, is_train=True)
#     train_size = int(0.8 * len(full_dataset))
#     val_size = len(full_dataset) - train_size
#     train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
#     val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

#     model = CNN1D(num_classes=2).to(device)

#     # Define loss function and optimizer
#     criterion = nn.MSELoss()
#     optimizer = optim.Adam(model.parameters(), lr=learning_rate)


#     for epoch in range(epochs):
#         model.train()
#         running_loss = 0.0

#         for inputs, targets in train_loader:
#             inputs, targets = inputs.to(device), targets.to(device)

#             optimizer.zero_grad()
#             outputs = model(inputs)
#             loss = criterion(outputs, targets)
#             loss.backward()
#             optimizer.step()

#             running_loss += loss.item() * inputs.size(0)

#         epoch_loss = running_loss / len(train_loader.dataset)

#         model.eval()
#         val_loss = 0.0
#         with torch.no_grad():
#             for inputs, targets in val_loader:
#                 inputs, targets = inputs.to(device), targets.to(device)
#                 outputs = model(inputs)
#                 loss = criterion(outputs, targets)
#                 val_loss += loss.item() * inputs.size(0)
        
#         epoch_val_loss = val_loss / len(val_loader.dataset)

#         if (epoch + 1) % 5 == 0 or epoch == 0:
#             print(f'Epoch [{epoch + 1}/{epochs}], Loss: {epoch_loss:.4f}, Val Loss: {epoch_val_loss:.4f}')

# if __name__ == "__main__":
#     main()



In [35]:
def main():
    data_path = 'data/final_data_outliers_clean_Iso.csv'
    batch_size = 64
    epochs = 150
    learning_rate = 0.001

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    df = pd.read_csv(data_path)
    # Filter out rows where 'Label' is 'nothing' and either 'Value_R' or 'Value_C' is greater than 0
    df = df[(df['Label'] != 'nothing') & ((df['Value_R'] > 0) | (df['Value_C'] > 0))].copy()

    """
    Split the dataset into training and validation sets using GroupShuffleSplit
    Avoid data leakage by ensuring that samples with the same 'Value_R' and 'Value_C' are in the same split
    """
    # df['Group_ID'] = df['Value_R'].astype(str) + "_" + df['Value_C'].astype(str)
    df['Group_ID'] = df['Label'] + "_" + df['Value_R'].astype(str) + "_" + df['Value_C'].astype(str)

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, val_idx = next(gss.split(df, groups=df['Group_ID']))

    train_df = df.iloc[train_idx].copy()
    val_df = df.iloc[val_idx].copy()

    #===================================================================================
    # FIT SCALERS 
    #===================================================================================
    mag_cols = [f'Magnitude_{i}' for i in range(15)]
    phase_cols = [f'Phase_{i}' for i in range(15)]

    scaler_mag = RobustScaler().fit(train_df[mag_cols].values)
    scaler_phase = RobustScaler().fit(train_df[phase_cols].values)
    scaler_target = RobustScaler().fit(train_df[['Value_R', 'Value_C']].values)

    # Transform
    X_train_mag = scaler_mag.transform(train_df[mag_cols].values)
    X_train_phase = scaler_phase.transform(train_df[phase_cols].values)
    Y_train = scaler_target.transform(train_df[['Value_R', 'Value_C']].values)

    X_val_mag = scaler_mag.transform(val_df[mag_cols].values)
    X_val_phase = scaler_phase.transform(val_df[phase_cols].values)
    Y_val = scaler_target.transform(val_df[['Value_R', 'Value_C']].values)

    # Load the dataset
    train_dataset = ImpedanceDataset(X_train_mag, X_train_phase, Y_train)
    val_dataset = ImpedanceDataset(X_val_mag, X_val_phase, Y_val)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = CNN1D_Robust().to(device)

    # Define loss function and optimizer
    # Fix point: Use AdamW optimizer and Huber loss for better performance on regression tasks
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    criterion = nn.HuberLoss()

    #Fix point: Scheduler & Early Stopping 
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    
    early_stopping_patience = 12
    epochs_no_improve = 0
    best_val_loss = float('inf')

    #===================================================================================
    # Training loop
    #===================================================================================

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)

        model.eval()
        val_loss = 0.0

    
        all_val_preds = []
        all_val_targets = []

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * inputs.size(0)

                # Collect predictions and targets for further analysis
                all_val_preds.append(outputs.cpu().numpy())
                all_val_targets.append(targets.cpu().numpy())
        
        epoch_val_loss = val_loss / len(val_loader.dataset)

        # Step the scheduler
        scheduler.step(epoch_val_loss)

        #Inverse transform the predictions and targets for evaluation
        val_preds_concat = np.vstack(all_val_preds)
        val_targets_concat = np.vstack(all_val_targets)

        real_preds = scaler_target.inverse_transform(val_preds_concat)
        real_targets = scaler_target.inverse_transform(val_targets_concat)

        mae_R = mean_absolute_error(real_targets[:, 0], real_preds[:, 0])
        mae_C = mean_absolute_error(real_targets[:, 1], real_preds[:, 1])

        print(f"Epoch {epoch+1:03d} | Train L: {epoch_loss:.4f} | Val L: {epoch_val_loss:.4f} || MAE_R: {mae_R:.1f} Ohm | MAE_C: {mae_C:.3f} C")
        
        # Early stopping logic
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), 'best_model.pth')
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= early_stopping_patience:
                print(f"\n[Early Stopping] Early stopping at Epoch {epoch+1}!")
                break
    return model, val_loader, scaler_target, device, val_df  

if __name__ == "__main__":
    model, val_loader, scaler_target, device, val_df = main()


Epoch 001 | Train L: 0.1533 | Val L: 0.0843 || MAE_R: 820.4 Ohm | MAE_C: 15.170 C
Epoch 002 | Train L: 0.1100 | Val L: 0.0705 || MAE_R: 759.5 Ohm | MAE_C: 13.169 C
Epoch 003 | Train L: 0.0938 | Val L: 0.0604 || MAE_R: 707.7 Ohm | MAE_C: 13.479 C
Epoch 004 | Train L: 0.0861 | Val L: 0.0472 || MAE_R: 599.9 Ohm | MAE_C: 10.784 C
Epoch 005 | Train L: 0.0785 | Val L: 0.0440 || MAE_R: 605.1 Ohm | MAE_C: 9.962 C
Epoch 006 | Train L: 0.0728 | Val L: 0.0430 || MAE_R: 593.0 Ohm | MAE_C: 10.500 C
Epoch 007 | Train L: 0.0713 | Val L: 0.0345 || MAE_R: 513.0 Ohm | MAE_C: 9.867 C
Epoch 008 | Train L: 0.0670 | Val L: 0.0298 || MAE_R: 488.4 Ohm | MAE_C: 7.992 C
Epoch 009 | Train L: 0.0637 | Val L: 0.0282 || MAE_R: 484.4 Ohm | MAE_C: 8.085 C
Epoch 010 | Train L: 0.0637 | Val L: 0.0271 || MAE_R: 459.6 Ohm | MAE_C: 8.188 C
Epoch 011 | Train L: 0.0617 | Val L: 0.0276 || MAE_R: 482.5 Ohm | MAE_C: 7.998 C
Epoch 012 | Train L: 0.0612 | Val L: 0.0284 || MAE_R: 455.1 Ohm | MAE_C: 8.685 C
Epoch 013 | Train L: 0.

In [43]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score

groups = val_df['Group_ID'].values

results_df = pd.DataFrame({
    'Group_ID': groups,
    'Actual_R': targets_real[:, 0],
    'Predicted_R': preds_real[:, 0],
    'Residual_R': preds_real[:, 0] - targets_real[:, 0],
    'AbsError_R': np.abs(preds_real[:, 0] - targets_real[:, 0]),
    'Actual_C': targets_real[:, 1],
    'Predicted_C': preds_real[:, 1],
    'Residual_C': preds_real[:, 1] - targets_real[:, 1],
    'AbsError_C': np.abs(preds_real[:, 1] - targets_real[:, 1]),
})

# Per-label (per R/C group) metrics
# overview_rows = []
# for group_id, sub in results_df.groupby('Group_ID'):
#     row = {'Group_ID': group_id, 'n_samples': len(sub)}
#     for target in ['R', 'C']:
#         actual = sub[f'Actual_{target}']
#         pred = sub[f'Predicted_{target}']
#         row[f'MAE_{target}'] = mean_absolute_error(actual, pred)
#         # R2 needs at least 2 samples with variance in the actual values
#         row[f'R2_{target}'] = r2_score(actual, pred) if len(sub) > 1 and actual.var() > 0 else np.nan
#     overview_rows.append(row)

# overview_df = pd.DataFrame(overview_rows).sort_values('Group_ID').reset_index(drop=True)

# # Overall metrics row appended at the top
# overall_row = {
#     'Group_ID': 'OVERALL',
#     'n_samples': len(results_df),
#     'MAE_R': mean_absolute_error(results_df['Actual_R'], results_df['Predicted_R']),
#     'R2_R': r2_score(results_df['Actual_R'], results_df['Predicted_R']),
#     'MAE_C': mean_absolute_error(results_df['Actual_C'], results_df['Predicted_C']),
#     'R2_C': r2_score(results_df['Actual_C'], results_df['Predicted_C']),
# }
# overview_df = pd.concat([pd.DataFrame([overall_row]), overview_df], ignore_index=True)

output_path = 'validation_predictions.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    # overview_df.to_excel(writer, sheet_name='Overview', index=False)
    results_df.to_excel(writer, sheet_name='Predictions', index=False)

print(f"Saved {len(results_df)} rows to {output_path}")


Saved 1710 rows to validation_predictions.xlsx


In [ ]:
# import pandas as pd

# results_df = pd.DataFrame({
#     'Actual_R': targets_real[:, 0],
#     'Predicted_R': preds_real[:, 0],
#     'Residual_R': preds_real[:, 0] - targets_real[:, 0],
#     'AbsError_R': np.abs(preds_real[:, 0] - targets_real[:, 0]),
#     'Actual_C': targets_real[:, 1],
#     'Predicted_C': preds_real[:, 1],
#     'Residual_C': preds_real[:, 1] - targets_real[:, 1],
#     'AbsError_C': np.abs(preds_real[:, 1] - targets_real[:, 1]),
# })

# output_path = 'validation_predictions.xlsx'
# results_df.to_excel(output_path, index=False, sheet_name='Predictions')

# print(f"Saved {len(results_df)} rows to {output_path}")
# results_df.head()

Saved 1710 rows to validation_predictions.xlsx


,Actual_R,Predicted_R,Residual_R,AbsError_R,Actual_C,Predicted_C,Residual_C,AbsError_C
0,1300.0,1285.301758,-14.698242,14.698242,68.0,63.970505,-4.029495,4.029495
1,1300.0,1355.353760,55.353760,55.353760,68.0,65.446259,-2.553741,2.553741
2,1300.0,1276.059448,-23.940552,23.940552,68.0,65.368027,-2.631973,2.631973
3,1300.0,1306.721802,6.721802,6.721802,68.0,64.605087,-3.394913,3.394913
4,1300.0,1360.978394,60.978394,60.978394,68.0,66.562378,-1.437622,1.437622
